# Office → HTML: `v3/` → `v3_html/`

Прогон `.docx` / `.xlsx` / `.xlsm`. Старые `.doc` / `.xls` роутятся (`legacy_format`), без конвертации.

```text
v3/docx/торг 12/file.docx  →  v3_html/docx/торг 12/file.html
v3/xlsx/кс-2/file.xlsx     →  v3_html/xlsx/кс-2/file.html
```

In [ ]:
from __future__ import annotations

import sys
import time
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "office_to_html.py").is_file():
    for p in [ROOT, *ROOT.parents]:
        if (p / "office_to_html.py").is_file():
            ROOT = p
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from office_normalize import PARSEABLE_OFFICE, SUPPORTED_OFFICE
from office_suitability import REASON_LABELS_RU
from office_to_html import office_to_html

V3_DIR = ROOT / "v3"
OUT_DIR = ROOT / "v3_html"

print(f"{ROOT.name}: {V3_DIR.name} → {OUT_DIR.name}")

## 1. Список файлов

In [ ]:
# парсим OOXML; .doc/.xls тоже найдём — уйдут в роут
OFFICE_EXTS = {e.lower() for e in SUPPORTED_OFFICE}


def discover_office_files(root: Path) -> list[Path]:
    return [
        p
        for p in sorted(root.rglob("*"))
        if p.is_file() and p.suffix.lower() in OFFICE_EXTS and not p.name.startswith(".~")
    ]


files = discover_office_files(V3_DIR)
by_ext = Counter(p.suffix.lower() for p in files)
parseable = sum(1 for p in files if p.suffix.lower() in PARSEABLE_OFFICE)
legacy = len(files) - parseable

print(f"{len(files)} files ({parseable} ooxml, {legacy} legacy→route)")
if by_ext:
    print(dict(by_ext))
for p in files:
    tag = "" if p.suffix.lower() in PARSEABLE_OFFICE else " [route]"
    print(f"  {p.relative_to(V3_DIR)}{tag}")

## 2. Прогон

Строка на файл: `[i/n] ok|route|err  path  type  accepted/total  sec`

In [ ]:
def out_path_for(src: Path) -> Path:
    return OUT_DIR / src.relative_to(V3_DIR).with_suffix(".html")


def run_v3(files: list[Path], *, skip_unsuitable: bool = True) -> list[dict]:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    rows: list[dict] = []
    n = len(files)
    if not n:
        print("empty")
        return rows

    print(f"run {n} → {OUT_DIR.name}/\n")
    t_all = time.perf_counter()

    for i, src in enumerate(files, start=1):
        rel = src.relative_to(V3_DIR)
        dst = out_path_for(src)
        dst.parent.mkdir(parents=True, exist_ok=True)
        prefix = f"[{i}/{n}]"

        try:
            result = office_to_html(src, dst, skip_unsuitable=skip_unsuitable)
        except Exception as exc:
            print(f"{prefix} err  {rel}  {type(exc).__name__}: {exc}")
            rows.append({"src": str(rel), "ok": False, "error": str(exc), "seconds": 0.0})
            continue

        if result.all_accepted:
            mark = "ok"
        elif result.accepted_pages > 0:
            mark = "part"
        else:
            mark = "route"

        types = ",".join(result.doc_types) if result.doc_types else "-"
        print(
            f"{prefix} {mark:5} {rel}  {types}  "
            f"{result.accepted_pages}/{result.total_pages}  {result.seconds:.2f}s"
        )
        if result.rejected_pages:
            codes = ",".join(sorted(result.reasons_summary()))
            print(f"         → {codes}")

        rows.append(
            {
                "src": str(rel),
                "dst": str(dst.relative_to(ROOT)),
                "ok": True,
                "mark": mark,
                "accepted": result.accepted_pages,
                "total": result.total_pages,
                "rejected": result.rejected_count,
                "doc_types": list(result.doc_types),
                "reasons": result.reasons_summary(),
                "warnings": list(result.warnings),
                "seconds": result.seconds,
            }
        )

    print(f"\ndone {time.perf_counter() - t_all:.1f}s")
    return rows


rows = run_v3(files)

## 3. Сводка

In [ ]:
def print_summary(rows: list[dict]) -> None:
    if not rows:
        print("no results")
        return

    ok_rows = [r for r in rows if r.get("ok")]
    err_rows = [r for r in rows if not r.get("ok")]
    total_units = sum(r.get("total", 0) for r in ok_rows)
    accepted_units = sum(r.get("accepted", 0) for r in ok_rows)
    rejected_units = sum(r.get("rejected", 0) for r in ok_rows)
    seconds = sum(r.get("seconds", 0.0) for r in ok_rows)

    type_counter: Counter[str] = Counter()
    reason_counter: Counter[str] = Counter()
    for r in ok_rows:
        for t in r.get("doc_types") or []:
            type_counter[t] += 1
        for code, n in (r.get("reasons") or {}).items():
            reason_counter[code] += n

    print("--- summary ---")
    print(f"files {len(ok_rows)}/{len(rows)}  err {len(err_rows)}")
    print(f"units {accepted_units}/{total_units} ok  route {rejected_units}")
    if ok_rows:
        print(f"time  {seconds:.1f}s  ({seconds / len(ok_rows):.2f}s/file)")
    if type_counter:
        print("types:", ", ".join(f"{t}={n}" for t, n in type_counter.most_common()))
    if reason_counter:
        print("route:", ", ".join(f"{c}×{n}" for c, n in reason_counter.most_common()))
        for code, n in reason_counter.most_common():
            print(f"  {code}: {REASON_LABELS_RU.get(code, code)}")
    for r in err_rows:
        print(f"err {r['src']}: {r.get('error')}")


print_summary(rows)

## 4. (Опционально) один файл

Раскомментируй и укажи путь относительно `v3/`.

In [ ]:
# ONE = V3_DIR / "docx" / "торг 12" / "torg12_sample.docx"
# if ONE.is_file():
#     r = office_to_html(ONE, out_path_for(ONE), quiet=False)
#     print(r.to_dict())
# else:
#     print("файл не найден:", ONE)